In [1]:
import fitz #pymupdf

In [2]:
file_path ="../data/raw/ncert_pdfs/iesc107.pdf"
doc=fitz.open(file_path)
print("No of pages = ",len(doc))

No of pages =  24


In [ ]:
def is_valid_heading(text):
    if not text:
        return False

    if text.isdigit():
        return False

    if len(text) > 80:
        return False

    if text.startswith(("Fig.", "Table", "Example", "Answer")):
        return False

    if text.startswith(("•", "✓", "y")):
        return False

    return True
def read_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    documents = []

    current_chapter_number = ""
    current_chapter_title = ""
    current_section = ""
    current_subsection = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text("text")

        blocks = page.get_text("dict")["blocks"]

        # Temporary title accumulator for this page
        chapter_title_parts = []
        section_parts = []
        subsection_parts = []

        for block in blocks:
            for line in block.get("lines", []):
                text = " ".join(
                    span["text"].strip()
                    for span in line["spans"]
                    if span["text"].strip()
                )

                if not text:
                    continue

                # Largest font in the line
                size = max(span["size"] for span in line["spans"])

                # Uncomment for debugging
                #print(f"{size:.1f} : {text}")

                # Chapter number (e.g. "7")
                if size >= 60:
                    current_chapter_number = text

                # Chapter title (can span multiple lines)
                elif size >= 25:
                    chapter_title_parts.append(text)

                # Ignore the word "Chapter"
                elif 20 <= size < 25 and text.lower() == "chapter":
                    continue
               
                # Section heading
                elif 14 <= size < 20:
                    if(is_valid_heading(text)):
                        section_parts.append(text)

                # Subsection / callout heading
                elif 12 <= size < 14:
                    if(is_valid_heading(text)):
                         subsection_parts.append(text)

                if section_parts:
                    current_section = " ".join(section_parts)

                if subsection_parts:
                    current_subsection = " ".join(subsection_parts)
                

        # Combine multi-line chapter title
        if chapter_title_parts:
            current_chapter_title = " ".join(chapter_title_parts).strip()

        documents.append({
            "content": page_text,
            "metadata": {
                "page": page_num + 1,
                "chapter_number": current_chapter_number,
                "chapter_title": current_chapter_title,
                "section": current_section,
                "callout": current_subsection,
                "source": pdf_path
            }
        })

    return documents

In [37]:
from gitsource import chunk_documents
documents = read_pdf(file_path)

doc_chunks = chunk_documents(documents,2000,1000)
doc_chunks

[{'start': 0,
  'content': 'Work, Energy, and \nSimple Machines\nChapter \n7\nIn earlier Chapters 4 and 6, you have learnt how forces change the \nmotion of objects, and how kinematic equations and Newton’s laws can \nbe used to analyse motion. But when forces change with time or act in \ncomplicated ways, applying these laws directly can become difficult. Is \nthere a simpler and more powerful way to understand such situations? \nIn this chapter, you will explore the ideas of work, energy and power, \nwhich often allow us to analyse motion and interactions more easily. You \nwill also learn about simple machines, which help us perform tasks with \nless effort and more convenience. These form the building blocks of many \neveryday machines. Energy, which is the capacity to do work, lies at the heart \nof all these ideas and of almost every activity in our daily life (Fig.\xa07.1).\nFig. 7.1: Energy required to carry out tasks comes from various sources\nFood provides \nenergy to walk\n